# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tanzina-Aranya-Islam/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [2]:
from google.colab import userdata
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")

path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

df_march = pd.read_parquet(
    path,
    storage_options={"token": HF_TOKEN}
)

print("Rows:", len(df_march))
print("Columns:", df_march.columns.tolist())

Rows: 9841378
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-05 — Build the initial feature vector

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "ga4_sessions",
    "gsc_avg_position",
    "sessions_ai"
]

X = df_march[feature_columns].copy()

print("Feature columns:")
print(X.columns.tolist())

print("\nFeature frame shape:", X.shape)

print("\nMissing values:")
print(X.isna().sum())

Feature columns:
['gsc_impressions', 'gsc_clicks', 'ga4_sessions', 'gsc_avg_position', 'sessions_ai']

Feature frame shape: (9841378, 5)

Missing values:
gsc_impressions           0
gsc_clicks                0
ga4_sessions        3018741
gsc_avg_position    6230317
sessions_ai         3018741
dtype: int64


## 2. Feature notes (meaning, missing, categorical, available-when?)

- gsc_impressions: Google Search Console impressions observed for the page. There are no missing values in the March 2026 slice. This is an observed feature available before the decision point.

- gsc_clicks: Google Search Console clicks observed for the page. There are no missing values in the March 2026 slice. This is an observed feature available before the decision point.

- ga4_sessions: GA4 sessions observed for the page. There are 3,018,741 missing values. I will treat missing values as unavailable GA4 data rather than automatically assuming zero sessions.

- gsc_avg_position: Average Google Search Console position observed for the page. There are 6,230,317 missing values. I will treat these as unavailable GSC data rather than assuming a numeric position.

- sessions_ai: AI referral sessions observed for the page. There are 3,018,741 missing values. I will treat missing values as unavailable AI referral data rather than automatically assuming zero sessions.

All five features are based on information observed in the March 2026 development window and are intended to be available before the defined decision point.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Feature availability summary:")

for col in feature_columns:
    missing = X[col].isna().sum()
    available = X[col].notna().sum()

    print(
        f"{col}: "
        f"available={available:,}, "
        f"missing={missing:,}"
    )

Feature availability summary:
gsc_impressions: available=9,841,378, missing=0
gsc_clicks: available=9,841,378, missing=0
ga4_sessions: available=6,822,637, missing=3,018,741
gsc_avg_position: available=3,611,061, missing=6,230,317
sessions_ai: available=6,822,637, missing=3,018,741


## 3. The leakage hunt

I checked the feature set for fields that could leak the target or a downstream decision. In particular, I checked the starter proxy label `trend_direction` and possible decision-output fields such as `health_score`, `priority_score`, and `action_type`.

I also checked that future-window information was not included in the March 2026 feature frame. The selected features come from the March 2026 observation window and are intended to be available before the decision point.

I will not use `trend_direction` as a feature because it is the proxy label. I will also exclude existing decision outputs if they are present. This keeps the feature vector independent from the target and downstream decisions.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

leakage_candidates = [
    "trend_direction",
    "health_score",
    "priority_score",
    "action_type"
]

print("Potential leakage fields:")

for col in leakage_candidates:
    if col in df_march.columns:
        print(f"{col}: FOUND")
    else:
        print(f"{col}: NOT FOUND")

print("\nCurrent feature set:")
print(feature_columns)

print("\nFuture-window check:")
print("Feature data range:",
      df_march["report_date"].min(),
      "to",
      df_march["report_date"].max())

Potential leakage fields:
trend_direction: NOT FOUND
health_score: NOT FOUND
priority_score: NOT FOUND
action_type: NOT FOUND

Current feature set:
['gsc_impressions', 'gsc_clicks', 'ga4_sessions', 'gsc_avg_position', 'sessions_ai']

Future-window check:
Feature data range: 2026-03-01 to 2026-03-31


## 4. What I excluded and why

- trend_direction: Excluded because it is the starter proxy label. Using it as a feature would leak the target.

- health_score: Excluded because it is an existing decision output if encountered. It could create circularity between the model input and an existing decision.

- priority_score: Excluded because it is a downstream decision score rather than an independent input signal.

- action_type: Excluded because it represents a downstream recommendation and could leak the decision being predicted.

- client_hash_id: Used only to identify and group client observations. It is not treated as a predictive feature.

- content_hash_id: Used only to identify content observations. It is not treated as a predictive feature.

- report_date: Used to define the observation period and prevent future-window leakage, but it is not used directly as a predictive feature.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded_fields = [
    "trend_direction",
    "health_score",
    "priority_score",
    "action_type",
    "client_hash_id",
    "content_hash_id",
    "report_date"
]

print("Excluded fields and whether they exist in the March dataset:")

for col in excluded_fields:
    print(f"{col}: {col in df_march.columns}")

print("\nFinal feature vector:")
print(feature_columns)

Excluded fields and whether they exist in the March dataset:
trend_direction: False
health_score: False
priority_score: False
action_type: False
client_hash_id: True
content_hash_id: True
report_date: True

Final feature vector:
['gsc_impressions', 'gsc_clicks', 'ga4_sessions', 'gsc_avg_position', 'sessions_ai']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.